# Crypto Price Prediction with PySpark
This notebook demonstrates how to predict BTC/ETH price movements (up/down) using PySpark MLlib with logistic regression and decision trees. We'll use a 1.22 GB dataset of 5-minute candles with various technical indicators.

In [1]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.classification import LogisticRegression, DecisionTreeClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when
import uuid

In [2]:
# Initialize Spark session with memory configuration for large dataset
spark = SparkSession.builder \
    .appName("CryptoPricePrediction") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()

# Show Spark version to confirm setup
print(f"Spark Version: {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/08 20:25:13 WARN Utils: Your hostname, sobreiro-msi, resolves to a loopback address: 127.0.1.1; using 172.16.21.134 instead (on interface wlo1)
26/05/08 20:25:13 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/08 20:25:14 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/08 20:25:14 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/05/08 20:25:14 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/05/08 20:25:14 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark Version: 4.1.1


In [4]:
# Load the 1.22 GB CSV dataset
data_path = './../data/btc_04h_usdt_binance.csv'
df = spark.read.option("header", "true").csv(data_path)

# Show first few rows
df.show(5)

# Print schema to understand data types
df.printSchema()

26/05/08 20:25:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------------------+-------+-------+-------+-------+----------+--------------------+----------------+------+--------------+---------------+------+-------+-------------------+------------------+----------+---------+------------------+-------------+-------------------+-----------+----------+------------------+--------------+--------------+--------------+--------------+--------------+---------------+---------------+--------------+------------------+-----------------+--------------+-------------------+---------------+---------------+--------------+--------------+--------------+--------------+--------------+--------------+-------------+----------+-----------------+---------------+--------------+--------------+--------------+--------------+--------------------+--------------------+---------------------+----------+----------------+---------+---------+-------------+--------------+-------------------+-------------------+----------------+------------------+---------+---------+-------------+----

In [5]:
# Define feature columns (technical indicators and other metrics)
features = [
    'open', 'high', 'low', 'volume', 'quote_vol', 'trades', 'taker_buy_base', 
    'taker_buy_quote', 'volume_adi', 'volume_obv', 'volume_cmf', 
    'volume_em', 'volume_sma_em', 'volume_vpt', 'volume_vwap', 'volume_mfi', 
    'volume_nvi', 'volatility_bbm', 'volatility_bbh', 'volatility_bbl', 
    'volatility_bbw', 'volatility_bbp', 'volatility_bbhi', 'volatility_bbli', 
    'volatility_kcc', 'volatility_kch', 'volatility_kcl', 'volatility_kcw', 
    'volatility_kcp', 'volatility_kchi', 'volatility_kcli', 'volatility_dcl', 
    'volatility_dch', 'volatility_dcm', 'volatility_dcw', 'volatility_dcp', 
    'volatility_atr', 'volatility_ui', 'trend_macd', 'trend_macd_signal', 
    'trend_macd_diff', 'trend_sma_fast', 'trend_sma_slow', 'trend_ema_fast', 
    'trend_ema_slow', 'trend_vortex_ind_pos', 'trend_vortex_ind_neg', 
    'trend_vortex_ind_diff', 'trend_trix', 'trend_mass_index', 'trend_dpo', 
    'trend_kst', 'trend_kst_sig', 'trend_kst_diff', 'trend_ichimoku_conv', 
    'trend_ichimoku_base', 'trend_ichimoku_a', 'trend_ichimoku_b', 'trend_stc', 
    'trend_adx', 'trend_adx_pos', 'trend_adx_neg', 'trend_cci', 
    'trend_visual_ichimoku_a', 'trend_visual_ichimoku_b', 'trend_aroon_up', 
    'trend_aroon_down', 'trend_aroon_ind', 'momentum_rsi', 'momentum_stoch_rsi', 
    'momentum_stoch_rsi_k', 'momentum_stoch_rsi_d', 'momentum_tsi', 
    'momentum_uo', 'momentum_stoch', 'momentum_stoch_signal', 'momentum_wr', 
    'momentum_ao', 'momentum_roc', 'momentum_ppo', 'momentum_ppo_signal', 
    'momentum_ppo_hist', 'momentum_kama', 'others_dr', 'others_dlr', 'others_cr', 
    'morningstar', 'hammer', 'piercing', '3soldiers', 'engulfing', 'sma200', 
    'sma50', 'ema200', 'ema50', 'slope', 'slope_obv', 'slope_rsi'
]

In [6]:
# Define feature columns (technical indicators and other metrics)
features = [
    'volume', 'quote_vol', 'trades', 'volume_obv',  'volume_vwap', 'volume_mfi', 
    'volume_nvi', 'volatility_bbm', 'volatility_bbh', 'volatility_bbl', 
    'volatility_bbw', 'volatility_bbp', 'volatility_bbhi', 'volatility_bbli', 
    'volatility_kcc', 'volatility_kch', 'volatility_kcl', 'volatility_kcw', 
    'volatility_kcp', 'volatility_kchi', 'volatility_kcli', 'volatility_dcl', 
    'volatility_dch', 'volatility_dcm', 'volatility_dcw', 'volatility_dcp', 
    'volatility_atr', 'volatility_ui', 'trend_macd', 'trend_macd_signal', 
    'trend_macd_diff', 'trend_sma_fast', 'trend_sma_slow', 'trend_ema_fast', 
    'trend_ema_slow', 'trend_vortex_ind_pos', 'trend_vortex_ind_neg', 
    'trend_vortex_ind_diff', 'trend_trix', 'trend_mass_index', 'trend_dpo', 
    'trend_kst', 'trend_kst_sig', 'trend_kst_diff', 'trend_ichimoku_conv', 
    'trend_ichimoku_base', 'trend_ichimoku_a', 'trend_ichimoku_b', 'trend_stc', 
    'trend_adx', 'trend_adx_pos', 'trend_adx_neg', 'trend_cci', 
    'trend_visual_ichimoku_a', 'trend_visual_ichimoku_b', 'trend_aroon_up', 
    'trend_aroon_down', 'trend_aroon_ind', 'momentum_rsi', 'momentum_stoch_rsi', 
    'momentum_stoch_rsi_k', 'momentum_stoch_rsi_d', 'momentum_tsi', 
    'momentum_uo', 'momentum_stoch', 'momentum_stoch_signal', 'momentum_wr', 
    'momentum_ao', 'momentum_roc', 'momentum_ppo', 'momentum_ppo_signal', 
    'momentum_ppo_hist', 'momentum_kama', 'others_dr', 'others_dlr', 'others_cr', 
    'morningstar', 'hammer', 'piercing', '3soldiers', 'engulfing', 'sma200', 
    'sma50', 'ema200', 'ema50', 'slope', 'slope_obv', 'slope_rsi'
]

In [7]:
# Create target variable: 1 if next candle's close > current close, else 0
from pyspark.sql.window import Window
from pyspark.sql.functions import lead

# Define a window to order rows by open_time (or another timestamp column)
window_spec = Window.orderBy("open_time")

# Create the next_close column by shifting the close column forward by 1
df = df.withColumn("next_close", lead(col("close").cast("double"), 1).over(window_spec))

# Create the label: 1 if next close > current close, else 0
df = df.withColumn("label", when(col("next_close") > col("close"), 1).otherwise(0))

# Drop rows with null values in features or label
df = df.dropna(subset=features + ["label"])

# Cast features to double for ML
for feature in features:
    df = df.withColumn(feature, col(feature).cast("double"))

# Show sample of processed data
df.select("close", "next_close", "label").show(5)

26/05/08 20:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+-------+----------+-----+
|  close|next_close|label|
+-------+----------+-----+
|3969.01|   3965.01|    0|
|3965.01|   3890.01|    0|
|3890.01|   3910.04|    1|
|3910.04|   3875.93|    0|
|3875.93|   3913.89|    1|
+-------+----------+-----+
only showing top 5 rows


In [8]:
# Split data into training (80%) and test (20%) sets
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Cache datasets for faster processing
train_df.cache()
test_df.cache()

# Print dataset sizes
print(f"Training set size: {train_df.count()} rows")
print(f"Test set size: {test_df.count()} rows")

26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:51 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 2

Training set size: 14891 rows


26/05/08 20:25:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
26/05/08 20:25:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


Test set size: 3647 rows


In [9]:
# Create feature vector
assembler = VectorAssembler(inputCols=features, outputCol="features")

# Scale features to normalize data
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", 
                       withStd=True, withMean=True)

In [10]:
# Initialize models
# Logistic Regression
lr = LogisticRegression(featuresCol="scaled_features", labelCol="label", 
                       maxIter=100, regParam=0.1)

# Decision Tree
dt = DecisionTreeClassifier(featuresCol="scaled_features", labelCol="label", 
                           maxDepth=5, seed=42)

In [11]:
# Create pipelines for both models
lr_pipeline = Pipeline(stages=[assembler, scaler, lr])
dt_pipeline = Pipeline(stages=[assembler, scaler, dt])

# Train models
print("Training Logistic Regression...")
lr_model = lr_pipeline.fit(train_df)

print("Training Decision Tree...")
dt_model = dt_pipeline.fit(train_df)

Training Logistic Regression...


Training Decision Tree...


In [12]:
# Make predictions on test set
lr_predictions = lr_model.transform(test_df)
dt_predictions = dt_model.transform(test_df)

# Show sample predictions
lr_predictions.select("label", "prediction", "probability").show(5)
dt_predictions.select("label", "prediction", "probability").show(5)

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    1|       1.0|[0.43163487420038...|
|    1|       1.0|[0.47740532713553...|
|    0|       1.0|[0.46107737201637...|
|    0|       1.0|[0.44866507478654...|
|    1|       1.0|[0.43120355046014...|
+-----+----------+--------------------+
only showing top 5 rows
+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|    1|       1.0|[0.25213675213675...|
|    1|       1.0|[0.43219264892268...|
|    0|       1.0|[0.25213675213675...|
|    0|       1.0|[0.46835443037974...|
|    1|       1.0|[0.46835443037974...|
+-----+----------+--------------------+
only showing top 5 rows


In [13]:
# Evaluate models using Area Under ROC Curve (AUC)
evaluator = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

lr_auc = evaluator.evaluate(lr_predictions)
dt_auc = evaluator.evaluate(dt_predictions)

# Print evaluation results
print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"Decision Tree AUC: {dt_auc:.4f}")

Logistic Regression AUC: 0.5321
Decision Tree AUC: 0.4990


In [14]:
# Evaluate models using Area Under ROC Curve (AUC) and Confusion Matrix
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.sql.functions import col

# AUC Evaluator
evaluator_roc = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")

# Multi-class evaluator for precision, recall, and F1-score
evaluator_multi = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction")

# Evaluate Logistic Regression
lr_auc = evaluator_roc.evaluate(lr_predictions)
lr_precision = evaluator_multi.evaluate(lr_predictions, {evaluator_multi.metricName: "weightedPrecision"})
lr_recall = evaluator_multi.evaluate(lr_predictions, {evaluator_multi.metricName: "weightedRecall"})
lr_f1 = evaluator_multi.evaluate(lr_predictions, {evaluator_multi.metricName: "f1"})

# Evaluate Random Forest
dt_auc = evaluator_roc.evaluate(dt_predictions)
dt_precision = evaluator_multi.evaluate(dt_predictions, {evaluator_multi.metricName: "weightedPrecision"})
dt_recall = evaluator_multi.evaluate(dt_predictions, {evaluator_multi.metricName: "weightedRecall"})
dt_f1 = evaluator_multi.evaluate(dt_predictions, {evaluator_multi.metricName: "f1"})

# Compute Confusion Matrix for Logistic Regression
lr_confusion = lr_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
print("Logistic Regression Confusion Matrix:")
lr_confusion.show()

# Compute Confusion Matrix for Random Forest
dt_confusion = dt_predictions.groupBy("label", "prediction").count().orderBy("label", "prediction")
print("Random Forest Confusion Matrix:")
dt_confusion.show()

# Print evaluation results
print(f"Logistic Regression AUC: {lr_auc:.4f}")
print(f"Logistic Regression Precision: {lr_precision:.4f}")
print(f"Logistic Regression Recall: {lr_recall:.4f}")
print(f"Logistic Regression F1-Score: {lr_f1:.4f}")
print(f"Random Forest AUC: {dt_auc:.4f}")
print(f"Random Forest Precision: {dt_precision:.4f}")
print(f"Random Forest Recall: {dt_recall:.4f}")
print(f"Random Forest F1-Score: {dt_f1:.4f}")

Logistic Regression Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|  642|
|    0|       1.0| 1113|
|    1|       0.0|  609|
|    1|       1.0| 1283|
+-----+----------+-----+

Random Forest Confusion Matrix:
+-----+----------+-----+
|label|prediction|count|
+-----+----------+-----+
|    0|       0.0|  938|
|    0|       1.0|  817|
|    1|       0.0|  888|
|    1|       1.0| 1004|
+-----+----------+-----+

Logistic Regression AUC: 0.5321
Logistic Regression Precision: 0.5248
Logistic Regression Recall: 0.5278
Logistic Regression F1-Score: 0.5160
Random Forest AUC: 0.4990
Random Forest Precision: 0.5332
Random Forest Recall: 0.5325
Random Forest F1-Score: 0.5327


In [15]:
# Stop Spark session to free resources
spark.stop()